# 7.4 Your turn: our relationship as a map of places

> **Question:** what does our relationship look like from a geographical point of view,
> going by our WhatsApp chat?

**My expectation:** during the long distance we talk about many cities, in France and in
the Netherlands. When I am pregnant, the cities stop coming up.

The method, from 7.1–7.3:

1. **Nodes are places**: every city (and Corsica) that came up in the chat, from a list I
   made by hand (`settings.PLACES`), coloured by country: France, the Netherlands, elsewhere.
2. **The edge rule, as a sentence:** *two places are connected when both came up on the
   same day.* That is 07.1's "who was nearby", with a whole day as the window, on places
   instead of people.
3. **Everything is counted in days**, not messages: ten messages about one train to
   Marseille are one day of Marseille. A dot grows with the days its place came up; a line
   gets heavier with the days the two places shared.
4. **The x-axis is time.** A place sits at the *median* day it came up, so its position says
   when we typically talked about it. The y-axis only separates the countries.


In [ ]:
import networkx as nx
import pandas as pd
from goad_toolkit.datatransforms import Pipeline
from goad_toolkit.visualizer import LinePlot, PlotSettings

from roos_pipeline import settings
from roos_pipeline.places import (
    TimelineGraphPlot,
    UnmergeMessages,
    lane_layout,
    place_days,
    place_graph,
    prune,
)
from roos_pipeline.settings import Paths

PREGNANCY = (settings.PREGNANCY_START, settings.BIRTH_DATE, "pregnancy")
MARGIN = pd.Timedelta(days=15)  # a little room left and right of the period

## 7.4.1 Messages back apart

From `1_messages.parquet` (the pipeline's stage 1: one author name per person, local time).
After an "afbeelding weggelaten" line the export sometimes glues the next messages into the
same row, each behind its own `[13-08-2023, 14:37:06] Roos:` header, sometimes a header from
another day. In lesson 6 `StripQuotedReply` cut those off. Here `UnmergeMessages` splits them
back into their own rows, with their own time, because a place in them came up on *that* day.


In [ ]:
raw = pd.read_parquet(Paths().messages)
messages = Pipeline().add(UnmergeMessages).apply(raw.copy())
messages = messages[messages["message"] != ""]
print(f"{len(raw):,} rows in the export -> {len(messages):,} messages after splitting, "
      f"{messages['ts_local'].min():%d %b %Y} to {messages['ts_local'].max():%d %b %Y}")

## 7.4.2 Which places came up

`place_days` checks every message against every place in `settings.PLACES` and keeps one row
per (day, place). The list is hand-made: I went through all capitalised words in the chat and
checked the ambiguous ones. Left out on purpose, so they count as *nothing*, not as a place:
"Nice" (almost always the English word), "Tours" ("deux tours"), "Spa", "Houten" ("houten
deur"), Luxembourg (the country), Schiphol (an airport) and neighbourhoods like Montmartre or
Overvecht, which belong to a city already on the list.


In [ ]:
days = place_days(messages)
per_place = days.groupby(["country", "place"]).size().rename("days").reset_index()
print(f"{days['place'].nunique()} places came up, on {days['day'].nunique()} different days\n")
for country in settings.COUNTRY_ORDER:
    top = per_place[per_place["country"] == country].nlargest(8, "days")
    print(f"{country:12s}", ", ".join(f"{p} {d}" for p, d in zip(top["place"], top["days"])))

## 7.4.3 The graph of the long-distance period

The long-distance period: from our first message (12 Aug 2023) to Louis's move to Utrecht
(`settings.MOVE_DATE`, 1 Nov 2024). The grey area in the plots is the pregnancy, from
`settings.PREGNANCY_START` (due-date convention, 41 weeks before Benji) to his birth. So in
this period, only its first three months are visible.

Before the picture, the numbers (07.1.8: *a network diagram is very good at looking like a
finding*).


In [ ]:
long_distance = place_graph(days, settings.LONG_DISTANCE_START, settings.MOVE_DATE)

strongest = sorted(long_distance.edges(data="weight"), key=lambda e: -e[2])[:8]
once = sum(1 for _, _, w in long_distance.edges(data="weight") if w == 1)
print(f"{long_distance.number_of_nodes()} places, {long_distance.number_of_edges()} links, "
      f"density {nx.density(long_distance):.2f}")
print(f"links that rest on a single day: {once} of {long_distance.number_of_edges()} "
      f"({once / long_distance.number_of_edges():.0%})\n")
print("the strongest links (days together):")
for u, v, w in strongest:
    print(f"  {u} - {v}: {w}")

## 7.4.4 Visualisation 1: every place

Every place that came up in the long-distance period, every link. Names only for places of
at least 10 days; the smaller dots are still there.


In [ ]:
ld_xlim = (settings.LONG_DISTANCE_START - MARGIN, settings.MOVE_DATE + MARGIN)

all_places = PlotSettings(
    figsize=(14, 6.5),
    title="Every place that came up on the same day, Aug 2023 - Nov 2024 (long distance)",
)
fig, ax = TimelineGraphPlot(all_places).plot(
    graph=long_distance, pos=lane_layout(long_distance), shade=PREGNANCY, label_min=10, xlim=ld_xlim
)

## 7.4.5 Visualisation 2: clutter reduced

My rules (`settings.MIN_PLACE_DAYS`, `MIN_LINK_DAYS`, `MAX_LINKS`, `ALWAYS_KEEP`), applied by
`prune` in this order:

1. **Places:** a place stays if it came up on at least 10 days. Exception: Corsica, the island
   where we met, always stays.
2. **Weak links:** a link stays if the two places came up together on at least 5 days.
3. **The 3 strongest links per place:** a link stays if it is in the top 3 of *at least one*
   of its places. So every small place keeps its link to its hub, and a hub like Utrecht can
   end up with more than 3.
4. **Nobody alone:** a place that has no link left gets back its single strongest one, even
   below 5 days. That is how Corsica (4 days with Paris) stays connected.

When two links are equally strong, the one to the place that came up on more days wins. The
order matters: step 3 ranks only the links left after steps 1 and 2.


In [ ]:
steps = {
    "start": long_distance,
    f"1. places >= {settings.MIN_PLACE_DAYS} days (+ Corsica)": prune(long_distance, min_link=0, max_links=10_000, no_loners=False),
    f"2. links >= {settings.MIN_LINK_DAYS} days": prune(long_distance, max_links=10_000, no_loners=False),
    f"3. top {settings.MAX_LINKS} of one of its places": prune(long_distance, no_loners=False),
    "4. nobody alone": prune(long_distance),
}
for label, graph in steps.items():
    alone = sum(1 for n in graph if graph.degree(n) == 0)
    print(f"{label:34s} {graph.number_of_nodes():3d} places, {graph.number_of_edges():3d} links, "
          f"{alone:2d} alone, {nx.number_connected_components(graph)} piece(s)")

reduced = steps["4. nobody alone"]
print()
for place in ["Corsica", "Terschelling"]:
    links = ", ".join(f"{other} ({reduced[place][other]['weight']} days)" for other in reduced[place])
    print(f"{place}, {reduced.nodes[place]['days']} days: linked to {links}")

In [ ]:
clean = PlotSettings(
    figsize=(14, 6.5),
    title="During our long distance relationship there were multiple places we called home",
)
fig, ax = TimelineGraphPlot(clean).plot(
    graph=reduced, pos=lane_layout(reduced), shade=PREGNANCY, xlim=ld_xlim
)

### Why rule 3 is the loose version

My first version was strict: a link only stayed if it was in the top 3 of *both* its places,
so no place could have more than 3. That left only the triangle Paris–Utrecht–Amsterdam plus
Rotterdam and Marseille, and most places stood alone: Utrecht's top 3 was full, so
Zoetermeer's strongest link (to Utrecht) was dropped. The price of the loose version: the
hubs now have more than 3 links.


In [ ]:
strict = prune(long_distance, both_ends=True, no_loners=False)
print(f"strict (top 3 of both places): {strict.number_of_edges():2d} links, "
      f"{sum(1 for n in strict if strict.degree(n) == 0)} places alone")
print(f"loose + nobody alone:          {reduced.number_of_edges():2d} links, "
      f"Paris {reduced.degree('Paris')}, Utrecht {reduced.degree('Utrecht')}, "
      f"Amsterdam {reduced.degree('Amsterdam')}")

## 7.4.6 Do the cities really stop during the pregnancy?

That was my expectation, and maybe the title. The network can't answer it well: a dot sits at
the *median* day of its place, so Paris and Utrecht end up in the middle of the period however
the mentions are spread. So first a check outside the network: **how many place-days per
month** (one place on one day = 1).


In [ ]:
per_month = (
    days.assign(month=days["day"].dt.to_period("M").dt.start_time)
    .groupby("month").size().rename("place_days").reset_index()
)

check = PlotSettings(
    figsize=(12, 4),
    title="Places that came up, per month",
    xlabel="",
    ylabel="place-days per month",
)
fig, ax = LinePlot(check).plot(data=per_month, x="month", y="place_days", marker="o", color=settings.GREY)
ax.axvspan(settings.PREGNANCY_START, settings.BIRTH_DATE, color=settings.LIGHT_GREY, alpha=0.45, lw=0)
ax.axvline(settings.MOVE_DATE, color=settings.DARK_GREY, linestyle="--", linewidth=1)
top = per_month["place_days"].max()
ax.annotate(" Louis moves", (settings.MOVE_DATE, top), color=settings.DARK_GREY)
ax.annotate(" pregnancy", (settings.PREGNANCY_START, top * 0.9), style="italic", color=settings.DARK_GREY)


def mean_per_month(start, end):
    window = per_month[(per_month["month"] >= start) & (per_month["month"] < end)]
    return window["place_days"].mean()


periods = {
    "long distance, before the pregnancy": (settings.LONG_DISTANCE_START, settings.PREGNANCY_START),
    "long distance, first months pregnant": (settings.PREGNANCY_START, settings.MOVE_DATE),
    "living together, rest of the pregnancy": (settings.MOVE_DATE, settings.BIRTH_DATE),
    "after Benji": (settings.BIRTH_DATE, per_month["month"].max() + pd.Timedelta(days=1)),
}
for label, (start, end) in periods.items():
    print(f"{label:40s} {mean_per_month(start, end):5.1f} place-days a month")

**What this shows** (check it against the numbers above after a re-run): in the first
three months of the pregnancy, places come up as often as before. The drop comes in
November 2024, when Louis moves to Utrecht and we no longer have to plan trains between Paris
and the Netherlands. So "the cities stop when I'm pregnant" is not what the chat says; "the
cities stop when we live together" fits better. Two things change close together, though
(pregnant in August, the move in November), so this is a description, not a cause.


## 7.4.7 Visualisation 3: long distance and the whole pregnancy

The longer period: from our first message to Benji's birth, so the grey area is complete.
Same clutter rules as visualisation 2; with even more days, the full graph only gets busier.


In [ ]:
longer = place_graph(days, settings.LONG_DISTANCE_START, settings.BIRTH_DATE)
longer_reduced = prune(longer)
print(f"all: {longer.number_of_nodes()} places, {longer.number_of_edges()} links -> "
      f"reduced: {longer_reduced.number_of_nodes()} places, {longer_reduced.number_of_edges()} links")
for name in longer_reduced:
    if name in long_distance:
        before, after = long_distance.nodes[name]["when"], longer.nodes[name]["when"]
        if abs(after - before) > pd.Timedelta(days=30):
            print(f"  {name}: median day {before:%b %Y} -> {after:%b %Y}")

longer_plot = PlotSettings(
    figsize=(15, 6.5),
    title="The strongest links between places, Aug 2023 - May 2025 (long distance and pregnancy)",
)
fig, ax = TimelineGraphPlot(longer_plot).plot(
    graph=longer_reduced,
    pos=lane_layout(longer_reduced),
    shade=PREGNANCY,
    xlim=(settings.LONG_DISTANCE_START - MARGIN, settings.BIRTH_DATE + MARGIN),
)
ax.axvline(settings.MOVE_DATE, color=settings.DARK_GREY, linestyle="--", linewidth=1, zorder=0)
_ = ax.annotate(" Louis moves", (settings.MOVE_DATE, -0.6), color=settings.DARK_GREY, fontsize=9)

## 7.4.8 What to write down, in my own words

The titles above only say *what* is drawn. The finding is for me to write.

1. **My edge rule, as a sentence**, including the window (a day). What can it *not* tell
   apart? (Think of one day where Paris and Barcelona came up together for reasons that have
   nothing to do with each other.)
2. **One number that is not a picture**: the density, the share of links resting on a single
   day, or the place-days per month.
3. **What I did about the nodes**: the hand-made place list, what I left out, Corsica kept
   against the rules. Would Corte and Piana belong *in* Corsica instead of next to it?
4. **The x-axis**: a dot sits at its median day. What does that hide for Paris, which came up
   all through the period? Would the *first* day suit the story I want to tell better?
5. **Rules 3 and 4**: why top 3 of *one* place, and why nobody alone? What do the links of
   Corsica and Terschelling (below 5 days) say, and what don't they?
6. **The title.** Does "the cities stop during the pregnancy" survive 7.4.6? If not, what is
   the sentence that does?
